# 03 · Data Fusion

Fuse all data sources to the zip × hurricane grain. Apply HUD crosswalk for tract-to-zip, geodesic distances via `pyproj.Geod`, and flood overlay in EPSG:5070.

In [1]:
import sys, os
from pathlib import Path
sys.path.append(str(Path.cwd().parent))
import pandas as pd, numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
pd.set_option('display.max_columns', 60)
from config import (
    DATA_PATHS, HURRICANE_META, STATES_IN_SCOPE,
    TARGET_COL, TARGET_CLASS_COL, FEATURE_GROUPS,
    RANDOM_STATE, SEVERITY_BINS, SEVERITY_LABELS,
)
RAW = DATA_PATHS['raw']; INTERIM = DATA_PATHS['interim']
PROC = DATA_PATHS['processed']; MODELS = DATA_PATHS['models']
OUT = DATA_PATHS['outputs']


In [2]:
import geopandas as gpd
from src.data_fusion import (
    merge_housing_assistance, tract_to_zip, snap_retailers_per_zip,
    build_track_linestring, compute_distance_to_track, pct_in_floodplain,
)

### Housing Assistance (Owners + Renters summed, per hurricane)

In [3]:
ha_frames = []
for h in HURRICANE_META:
    dn = h['disaster_number']
    o = pd.read_csv(RAW / f'fema_housing_owners_{dn}.csv')
    r = pd.read_csv(RAW / f'fema_housing_renters_{dn}.csv')
    ha_frames.append(merge_housing_assistance(o, r))
ha = pd.concat(ha_frames, ignore_index=True)
print('rows (zip × hurricane):', len(ha))

rows (zip × hurricane): 7229


### ACS demographics

In [4]:
acs = pd.read_csv(RAW / 'census_acs5_zcta.csv', dtype={'zip_code': str})
acs['zip_code'] = acs['zip_code'].str.zfill(5)
from src.feature_engineering import derive_demographic_shares
acs = derive_demographic_shares(acs)
print(acs.shape)

(33774, 27)


### CDC SVI — replace -999, then tract→zip via HUD

In [5]:
svi = pd.read_csv(RAW / 'cdc_svi_2022.csv', low_memory=False)
svi = svi.replace(-999, np.nan)
cw = pd.read_excel(RAW / 'hud_tract_zip.xlsx')
svi_keep = ['FIPS','RPL_THEME1','RPL_THEME2','RPL_THEME3','RPL_THEME4','RPL_THEMES']
svi_zip = tract_to_zip(svi[svi_keep], cw, tract_col='FIPS',
                       value_cols=svi_keep[1:])
svi_zip = svi_zip.rename(columns={
    'RPL_THEME1':'svi_socioeconomic','RPL_THEME2':'svi_household_comp',
    'RPL_THEME3':'svi_minority_lang','RPL_THEME4':'svi_housing_transport',
    'RPL_THEMES':'svi_overall'})
print(svi_zip.shape); svi_zip.head()

(39180, 6)


,zip_code,svi_socioeconomic,svi_household_comp,svi_minority_lang,svi_housing_transport,svi_overall
0,00501,0.000000,0.000000,0.000000,0.000000,0.000000
1,01001,0.365519,0.547502,0.239272,0.566116,0.438860
2,01002,0.417859,0.227482,0.443690,0.653897,0.430972
3,01003,0.482757,0.146292,0.455233,0.693876,0.446146
4,01004,0.622700,0.209500,0.411700,0.848000,0.620000


### USDA Food Access Atlas (tract → zip)

In [6]:
fa = pd.read_excel(RAW / 'food_access_atlas.xlsx', sheet_name='Food Access Research Atlas')
fa_keep = ['CensusTract','LILATracts_1And10','lalowi1','lahunv1share','TractSNAP']
fa_zip = tract_to_zip(fa[fa_keep], cw, tract_col='CensusTract', value_cols=fa_keep[1:])
fa_zip['food_desert_flag'] = (fa_zip['LILATracts_1And10'] >= 0.5).astype(int)
fa_zip = fa_zip.rename(columns={'TractSNAP':'snap_participation_pct'})
print(fa_zip.shape)

(35585, 6)


### SNAP retailers — spatial join to ZCTA

In [7]:
import glob
snap_csv = next((p for p in (RAW / 'snap_retailers').rglob('*.csv')), None)
snap = pd.read_csv(snap_csv)
zcta_path = next((RAW / 'zcta').rglob('*.shp'))
zcta_gdf = gpd.read_file(zcta_path)
snap_zip = snap_retailers_per_zip(snap, zcta_gdf)
print(snap_zip.shape); snap_zip.head()

(33791, 3)


,zip_code,snap_retailer_count,dist_nearest_supermarket_mi
0,00601,0,116.341963
1,00602,0,147.499727
2,00603,0,142.887845
3,00606,0,128.328099
4,00610,0,141.538819


### IBTrACS distance-to-track per hurricane (geodesic)

In [8]:
ibt = pd.read_csv(RAW / 'ibtracs_na.csv', low_memory=False, skiprows=[1])
dist_rows = []
for h in HURRICANE_META:
    track = build_track_linestring(ibt, h['name'], h['year'])
    # restrict zip set to states affected by this hurricane for speed
    zcta_sub = zcta_gdf  # for full rigor, filter by state
    d = compute_distance_to_track(zcta_sub, track)
    d['disaster_number'] = h['disaster_number']
    dist_rows.append(d)
dist = pd.concat(dist_rows, ignore_index=True)
print(dist.shape)

dist_to_track: 100%|██████████| 33791/33791 [00:01<00:00, 27529.87it/s]

(473074, 3)


### Flood overlay (EPSG:5070, county-by-county)

In [9]:
flood_rows = []
for st_fips in ['48','22','12','37','45','13','01','28']:
    p = RAW / f'nfhl_sfha_{st_fips}.geojson'
    if not p.exists(): continue
    nfhl = gpd.read_file(p)
    flood_rows.append(pct_in_floodplain(zcta_gdf, nfhl))
flood = pd.concat(flood_rows, ignore_index=True) if flood_rows else pd.DataFrame({'zip_code':[],'pct_in_100yr_floodplain':[]})
flood = flood.groupby('zip_code')['pct_in_100yr_floodplain'].max().reset_index()
print(flood.shape)

flood overlay by county: 100%|██████████| 1/1 [00:27<00:00, 27.33s/it]


(95, 2)


### FEMA NRI — tract-level hazard scores aggregated to ZIP
FEMA discontinued zip-level NRI in 2025; we download the tract-level CSV from the ArcGIS mirror and aggregate to ZIP via the same HUD crosswalk used for SVI and Food Atlas. Provides coastal-flood (`CFLD_RISKS`) and hurricane (`HRCN_RISKS`) risk scores.

In [10]:
nri_path = RAW / 'fema_nri_tract.csv'
if nri_path.exists():
    # Tract-level CSV (~85k US census tracts). FIPS column is 'TRACTFIPS'
    # (11-digit string). We aggregate to ZIP via the same HUD crosswalk
    # used for SVI and Food Atlas.
    nri = pd.read_csv(nri_path, dtype={'TRACTFIPS': str}, low_memory=False)
    keep_src = {'CFLD_RISKS': 'nri_cflood_score',
                'HRCN_RISKS': 'nri_hrcn_score'}
    keep_src = {k: v for k, v in keep_src.items() if k in nri.columns}
    if not keep_src:
        # Fallback: some NRI versions name columns CFLD_AFREQ / HRCN_AFREQ.
        # Use composite RISK_SCORE if hazard-specific columns are absent.
        keep_src = {c: f'nri_{c.lower()}' for c in ('RISK_SCORE',) if c in nri.columns}
    nri_t = nri[['TRACTFIPS'] + list(keep_src)].rename(columns=keep_src)
    nri_zip = tract_to_zip(nri_t, cw, tract_col='TRACTFIPS',
                           value_cols=list(keep_src.values()))
    print('NRI tract -> zip:', nri_zip.shape, 'cols:', list(nri_zip.columns))
else:
    print('[warn] fema_nri_tract.csv missing — re-run notebook 01 cell 2')
    nri_zip = pd.DataFrame({'zip_code': [], 'nri_cflood_score': [], 'nri_hrcn_score': []})

NRI tract -> zip: (38969, 3) cols: ['zip_code', 'nri_cflood_score', 'nri_hrcn_score']


### Merge everything

In [11]:
fused = (ha
    .merge(acs, on='zip_code', how='left')
    .merge(svi_zip, on='zip_code', how='left')
    .merge(fa_zip, on='zip_code', how='left')
    .merge(snap_zip, on='zip_code', how='left')
    .merge(dist, on=['zip_code','disaster_number'], how='left')
    .merge(flood, on='zip_code', how='left')
    .merge(nri_zip, on='zip_code', how='left')
)
fused['snap_retailers_per_1k'] = fused['snap_retailer_count'] / (fused['population'].replace(0, np.nan) / 1000)
print('fused shape:', fused.shape)
for c in ['nri_cflood_score','nri_hrcn_score']:
    if c in fused.columns:
        print(f'  {c} non-null: {fused[c].notna().mean()*100:.1f}%')
fused.to_csv(INTERIM / 'fused_zip_hurricane.csv', index=False)
fused.head()


fused shape: (7229, 57)
  nri_cflood_score non-null: 86.8%
  nri_hrcn_score non-null: 86.8%


,disaster_number,zip_code,state,county,total_inspected,total_major_substantial,total_no_damage,total_minor,total_moderate,total_approved_dollars,validRegistrations,repairReplaceAmount,rentalAmount,otherNeedsAmount,NAME,population,median_income,poverty_count,renters,male_65_66,male_67_69,male_70_74,male_75_79,male_80_84,male_85_over,female_65_66,female_67_69,female_70_74,female_75_79,female_80_84,female_85_over,white_alone,no_vehicle_households,mobile_homes,pct_poverty,pct_renters,pct_elderly_65plus,pct_minority,pct_no_vehicle,pct_mobile_homes,svi_socioeconomic,svi_household_comp,svi_minority_lang,svi_housing_transport,svi_overall,LILATracts_1And10,lalowi1,lahunv1share,snap_participation_pct,food_desert_flag,snap_retailer_count,dist_nearest_supermarket_mi,distance_to_track_km,pct_in_100yr_floodplain,nri_cflood_score,nri_hrcn_score,snap_retailers_per_1k
0,4283,06475,FL,St. Johns (County),1.0,0.0,1.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,ZCTA5 06475,10506.0,95795.0,448.0,780.0,129.0,180.0,342.0,305.0,161.0,141.0,121.0,214.0,368.0,386.0,291.0,288.0,10115.0,255.0,87.0,4.264230,7.424329,27.850752,3.721683,2.427184,0.828098,0.167204,0.548677,0.109494,0.434818,0.261500,NaN,NaN,NaN,NaN,NaN,20.0,0.0,667.434918,NaN,NaN,NaN,1.903674
1,4283,31264,FL,Flagler (County),0.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,4283,31268,FL,Volusia (County),0.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,4283,31558,FL,Nassau (County),0.0,0.0,1.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,ZCTA5 31558,21725.0,63849.0,3684.0,3707.0,94.0,319.0,429.0,221.0,171.0,39.0,271.0,258.0,663.0,357.0,119.0,99.0,16553.0,419.0,490.0,16.957422,17.063291,13.993096,23.806674,1.928654,2.255466,0.563199,0.502831,0.463505,0.383254,0.495713,0.175958,360.76243,0.331538,46.291809,0.0,33.0,0.0,87.173735,NaN,90.001495,92.677612,1.518987
4,4283,32007,FL,Putnam (County),0.0,0.0,0.0,0.0,0.0,0.0,2.0,0.0,0.0,0.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.00000,0.000000,0.000000,0.0,NaN,NaN,NaN,NaN,0.000000,0.000000,NaN
